# Ticker Usage Example

This notebook shows how to create a `Ticker`, plug in a sample Candle API provider, attach 5 indicators, initialize data from the API, and poll new candles.

In [1]:
from __future__ import annotations

import random
from typing import Any

from tradingbot.core.constants import Interval
from tradingbot.core.indicators import (
    AverageTrueRange,
    ExponentialMovingAverage,
    RelativeStrengthIndex,
    SimpleMovingAverage,
    VolumeWeightedAveragePrice,
)
from tradingbot.core.ticker import Ticker

In [2]:
class SampleCandleAPI:
    """Simple in-memory candle API generator for local testing."""

    def __init__(self, seed: int = random.randint(0, 10)) -> None:
        self._rng = random.Random(seed)
        self._last_timestamp = 1711700000
        self._last_close = 100.0

    def _next_candle(self) -> dict[str, Any]:
        drift = self._rng.gauss(0, 0.5)
        open_price = self._last_close
        close_price = open_price + drift
        high = max(open_price, close_price) + abs(self._rng.gauss(0, 0.3))
        low = min(open_price, close_price) - abs(self._rng.gauss(0, 0.3))
        volume = self._rng.uniform(900, 2000)

        candle = {
            "timestamp": self._last_timestamp,
            "open": round(open_price, 2),
            "high": round(high, 2),
            "low": round(low, 2),
            "close": round(close_price, 2),
            "volume": round(volume, 2),
        }

        self._last_timestamp += 60
        self._last_close = close_price
        return candle

    def fetch_candles(
        self,
        symbol: str,
        interval: str,
        limit: int,
    ) -> list[dict[str, Any]]:
        _ = (symbol, interval)
        return [self._next_candle() for _ in range(limit)]

In [3]:
# Add 5 simple indicators
indicators = [
    SimpleMovingAverage(period=5),
    ExponentialMovingAverage(period=9),
    VolumeWeightedAveragePrice(period=10),
    RelativeStrengthIndex(period=14),
    AverageTrueRange(period=14),
]

api = SampleCandleAPI()

ticker = Ticker(
    name="BTCUSDT",
    interval=Interval.MINUTE,
    indicators=indicators,
    candle_api=api,
)

ticker.initialize(api_limit=50, recompute=True)

print("Ticker:", ticker)
print("Candles loaded:", len(ticker.sequence.candles))
print("Latest indicator values:")
for name, value in ticker.get_latest_indicator_values().items():
    print(f"  {name}: {value}")

Ticker: Ticker(name='BTCUSDT', sequence_length=50, indicators=['MA5', 'EMA9', 'VWAP10', 'RSI14', 'ATR14'])
Candles loaded: 50
Latest indicator values:
  MA5: 101.146
  EMA9: 100.88193112193338
  VWAP10: 100.95382314501052
  RSI14: 64.41441441441444
  ATR14: 0.7614285714285712


In [5]:
# Poll two fresh candles from the API and auto-apply update/append logic
ticker.poll()

print("\nAfter poll:")
print("Candles in sequence:", len(ticker.sequence.candles))
print("Latest candle timestamp:", ticker.sequence.candles[-1].timestamp)
print("Latest indicator values:")
for name, value in ticker.get_latest_indicator_values().items():
    print(f"  {name}: {value}")


After poll:
Candles in sequence: 50
Latest candle timestamp: 1711705940
Latest indicator values:
  MA5: 99.366
  EMA9: 99.59168798539807
  VWAP10: 99.89794848688186
  RSI14: 24.203821656050764
  ATR14: 0.82642857142857
